# config

In [ ]:
config = {
    "file_config": {
        "input_path": ""
    },
    "db_config": {
      "opensearch": {
        "hosts": [
          {
            "host": "",
            "port": 80
          }
        ],
        "user": "",
        "password": "",
        "index": ""
      }
    },
    "custom_config": {
        "total_data_gb": 10080,
        "shard_size_gb": 40,
        "bulk_size": 10000,
        "read_partitions": 100,
        "write_partitions": 100,
        "recreate_index": False,
        "bulk_action": "create",
        "opensearch_progress_root": "",
        "skip_existing_doc_ids": True,
        "doc_id_lookup_batch_size": 0,
        "doc_id_delete_batch_size": 0,
        "doc_id_delete_request_timeout": 0,
        "test_sample_rows": 0,
        "enable_duplicate_sample": False
    }
}


In [ ]:
import os
os.environ["SPARK_USER"] = "renpengli"
os.environ.setdefault("XINGHE_CONF_DIR", "/share/renpengli/conf")

import json
import math
import socket
import time
import uuid
import logging
from datetime import datetime, timezone

from opensearchpy import OpenSearch
from pyspark import SparkConf
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql.functions import col, from_json, lower, transform, udf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, MapType, ArrayType, LongType

from xinghe.s3 import *
from xinghe.utils.json_util import *
from xinghe.spark import new_spark_session, read_any_path

# config = json_loads('${config}')
custom_config = config["custom_config"]
file_config = config["file_config"]
s3_config = config.get("s3_config", {})
db_config = config.get("db_config", {})

input_path = file_config["input_path"]
opensearch_config = db_config["opensearch"]
INPUT_PROGRESS_ROOT = custom_config.get("opensearch_progress_root", f"{str(input_path).rstrip('/')}/_opensearch_progress")
INPUT_PROGRESS_PATH = f"{INPUT_PROGRESS_ROOT.rstrip('/')}/progress.json"
DOC_ID_PROGRESS_ROOT = f"{INPUT_PROGRESS_ROOT.rstrip('/')}/doc_id_progress"
DOC_ID_PROBE_ROOT = f"{INPUT_PROGRESS_ROOT.rstrip('/')}/_doc_id_probe"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + f"-{uuid.uuid4().hex[:8]}"
CURRENT_DOC_ID_PROGRESS_PATH = f"{DOC_ID_PROGRESS_ROOT.rstrip('/')}/part-{RUN_ID}.jsonl"
CURRENT_DOC_ID_PROBE_ROOT = f"{DOC_ID_PROBE_ROOT.rstrip('/')}/{RUN_ID}"

def get_s3_args(path, s3_config):
    bucket = split_s3_path(path)[0]
    s3_args = s3_config.get(bucket, {})
    if not s3_args:
        s3_args = get_s3_config(path)
    print(f"bucket={bucket}, s3_args={s3_args}")
    return bucket, s3_args

read_s3_bucket, read_s3_config = get_s3_args(input_path, s3_config)

# --- 极限写入参数 ---
TOTAL_DATA_GB = custom_config.get("total_data_gb", 10080)
SHARD_SIZE_GB = custom_config.get("shard_size_gb", 40)
NUM_SHARDS = math.ceil(TOTAL_DATA_GB / SHARD_SIZE_GB)

BULK_SIZE = custom_config.get("bulk_size", 10_000)
READ_PARTITIONS = custom_config.get("read_partitions", 100)
WRITE_PARTITIONS = custom_config.get("write_partitions", 100)
# 建索引策略：测试期默认删掉同名索引后重建；如果想复用已有索引，改成 False。
RECREATE_INDEX = custom_config.get("recreate_index", True)
BULK_ACTION = str(custom_config.get("bulk_action", "index")).strip().lower() or "index"
if BULK_ACTION not in {"index", "create"}:
    raise ValueError(f"unsupported bulk_action: {BULK_ACTION}")
INDEX_DELETE_WAIT_SECONDS = 30
# 先抽样验证：默认只写 10 条。全量导入时改成 None 或 0。
TEST_SAMPLE_ROWS = custom_config.get("test_sample_rows", 10)
WRITE_PARTITIONS = max(1, min(WRITE_PARTITIONS, TEST_SAMPLE_ROWS or WRITE_PARTITIONS))
ENABLE_DUPLICATE_SAMPLE = custom_config.get("enable_duplicate_sample", False)
SKIP_EXISTING_DOC_IDS = bool(custom_config.get("skip_existing_doc_ids", True))
DOC_ID_LOOKUP_BATCH_SIZE = int(custom_config.get("doc_id_lookup_batch_size", 0) or 0)
DOC_ID_DELETE_BATCH_SIZE = int(custom_config.get("doc_id_delete_batch_size", 0) or 0)
DOC_ID_DELETE_REQUEST_TIMEOUT = int(custom_config.get("doc_id_delete_request_timeout", 0) or 0)

# --- OpenSearch 连接配置（按实际环境修改）---
OS_HOSTS = opensearch_config["hosts"]
OS_AUTH = (opensearch_config.get("user", ""), opensearch_config.get("password", ""))
OS_INDEX = opensearch_config.get("index", "")

OS_CONN = dict(hosts=OS_HOSTS, http_auth=OS_AUTH, use_ssl=False, verify_certs=False)

client = OpenSearch(**OS_CONN, timeout=60)

def load_progress_payload(path: str) -> dict:
    try:
        return json.loads(read_s3_object(path).read().decode('utf-8'))
    except Exception:
        return {}

def save_progress_payload(path: str, payload: dict) -> None:
    try:
        delete_s3_object(path, dry_run=False)
    except Exception:
        pass
    put_s3_object(path, json.dumps(payload, ensure_ascii=False, indent=2).encode('utf-8'))


In [ ]:
dolphin_spark_config = {
    # spark 资源配置
    "spark.driver.memory": "6g",
    "spark.executor.cores": "4",
    "spark.kubernetes.executor.limit.cores": "8",  # spark.kubernetes.executor.limit.cores 是对cpu的限制
    "spark.executor.memory": "8g",
    "spark.executor.instances": "25",
    # spark 其他配置
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.speculation": "true",
    "spark.default.parallelism": "8000",
    "spark.executorEnv.SPARK_USER": "renpengli",
    #sparksql 配置
    "spark.sql.sources.partitionOverwriteMode":"dynamic",
    "spark.sql.parquet.compression.codec":"snappy",
    "spark.sql.files.maxRecordsPerFile": 50000,
    "spark.sql.adaptive.enabled":"true",
    "spark.sql.shuffle.partitions":"4096",
    "spark.sql.adaptive.coalescePartitions.enabled":"true",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes":"256MB",
    "spark.sql.autoBroadcastJoinThreshold":"-1",
    "spark.sql.adaptive.shuffle.targetPostShuffleInputSize":"67108864",
    # hadoop s3a 配置
    "spark.hadoop.fs.s3a.impl":"org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.hadoop.fs.s3a.connection.ssl.enabled": "false",
    "spark.hadoop.fs.s3a.path.style.access": "true",
    "spark.hadoop.fs.s3a.endpoint": read_s3_config["endpoint"],
    "spark.hadoop.fs.s3a.access.key": read_s3_config["ak"],
    "spark.hadoop.fs.s3a.secret.key": read_s3_config["sk"],
    "spark.hadoop.fs.s3a.multiobjectdelete.enable":"false",
    "spark.hadoop.fs.s3a.directory.marker.retention":"keep",
    "spark.hadoop.fs.s3a.fast.upload":"true",
    "spark.hadoop.fs.s3a.connection.maximum":"1000",
    "spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version":"2",
    # kubernetes 配置
    "spark.kubernetes.executor.deleteOnTermination":"false",
    "spark.submit.deployMode": "cluster",
    "spark.kubernetes.namespace": "dataops-paas",
    "spark.kubernetes.authenticate.driver.serviceAccountName": "spark-default",
    "spark.kubernetes.container.image.pullPolicy": "Always",
    "spark.kubernetes.container.image": "registry.sensetime.com/hadoop/dataops/apache/spark:3.5.7-data-platform",
    # event log 配置
    "spark.eventLog.enabled": "true",
    "spark.eventLog.dir": "/spark/eventLog",
    # Magic Committer（推荐 - 不需要临时目录，性能最佳）
    # 参考: https://hadoop.apache.org/docs/current/hadoop-aws/tools/hadoop-aws/committers.html
    "spark.jars":"/share/spark-jars/spark-hadoop-cloud_2.12-3.5.7.jar",
    "spark.hadoop.fs.s3a.committer.name": "magic",
    "spark.hadoop.fs.s3a.committer.magic.enabled": "true" ,
    "spark.hadoop.mapreduce.outputcommitter.factory.scheme.s3a": "org.apache.hadoop.fs.s3a.commit.S3ACommitterFactory",
    "spark.sql.sources.commitProtocolClass": "org.apache.spark.internal.io.cloud.PathOutputCommitProtocol",
    "spark.sql.parquet.output.committer.class": "org.apache.spark.internal.io.cloud.BindingParquetOutputCommitter"
}

spark_config = {
    **dolphin_spark_config,
    "spark.kubernetes.executor.podTemplateFile": "/share/renpengli/pod-template/pod-template-dolphin.yaml",
    "spark.kubernetes.executor.label.queue": "root.datacenter.data-producer.default",
    "spark.driver.host": os.environ.get("POD_IP"),
    "spark.submit.deployMode": "client",
}

conf = SparkConf() 
conf.setAll(list(spark_config.items()))

# 初始化Spark
master = "k8s://https://{kubernetes_service_host}:{kubernetes_service_port}".format(kubernetes_service_host = os.environ.get("KUBERNETES_SERVICE_HOST"), 
        kubernetes_service_port = os.environ.get("KUBERNETES_SERVICE_PORT"))
tt = int(time.time())
spark = SparkSession.builder.master(master).config(conf=conf).appName(f"chunk_to_opensearch_{tt}").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark


# funcs

In [ ]:
def validate_os_hosts(hosts):
    if not hosts:
        raise ValueError("OS_HOSTS 不能为空")
    for item in hosts:
        host = str((item or {}).get("host") or "").strip()
        port = int((item or {}).get("port") or 0)
        if not host:
            raise ValueError(f"OS_HOSTS 配置缺少 host: {item!r}")
        if host == "your-opensearch-host":
            raise ValueError("OS_HOSTS 仍是占位符 your-opensearch-host，请改成真实 OpenSearch 域名或 IP")
        if port <= 0:
            raise ValueError(f"OS_HOSTS 配置缺少有效 port: {item!r}")
        try:
            socket.getaddrinfo(host, port, socket.AF_UNSPEC, socket.SOCK_STREAM)
        except socket.gaierror as exc:
            raise ValueError(
                f"无法解析 OpenSearch 主机名 {host!r}:{port}，请检查 OS_HOSTS、DNS、VPN 或内网连通性"
            ) from exc


def probe_opensearch(client):
    try:
        info = client.info()
    except Exception as exc:
        raise RuntimeError(f"OpenSearch 连通性探测失败: {exc}") from exc
    cluster_name = info.get("cluster_name") or ""
    version = ((info.get("version") or {}).get("number")) or ""
    print(f"OpenSearch reachable: cluster={cluster_name} version={version}")
    return info

def wait_until_index_absent(client, index, timeout_seconds=30):
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        if not client.indices.exists(index=index):
            return True
        time.sleep(1)
    return not client.indices.exists(index=index)


def utc_millis_to_iso_z(ms):
    if ms is None:
        return None
    ms = int(ms)
    if ms <= 0:
        return None
    dt = datetime.fromtimestamp(ms / 1000.0, tz=timezone.utc)
    return dt.strftime("%Y-%m-%dT%H:%M:%S.") + f"{dt.microsecond // 1000:03d}Z"

def parse_extra_info(value):
    if value is None:
        return None
    if isinstance(value, dict):
        return {str(k): None if v is None else str(v) for k, v in value.items()}
    s = str(value).strip()
    if not s:
        return None
    try:
        obj = json.loads(s)
    except Exception:
        return None
    if not isinstance(obj, dict):
        return None
    out = {}
    for k, v in obj.items():
        if v is None:
            out[str(k)] = None
        elif isinstance(v, (dict, list)):
            out[str(k)] = json.dumps(v, ensure_ascii=False)
        else:
            out[str(k)] = str(v)
    return out

utc_millis_to_iso_z_udf = udf(utc_millis_to_iso_z, StringType())
parse_extra_info_udf = udf(parse_extra_info, MapType(StringType(), StringType(), True))

def sample_duplicate_chunk_ids(client, index_name, size=10):
    if not ENABLE_DUPLICATE_SAMPLE:
        print(f"skip duplicate chunk_id sample for {index_name}: ENABLE_DUPLICATE_SAMPLE=False")
        return None
    try:
        resp = client.search(
        index=index_name,
        body={
            "size": 0,
            "aggs": {
                "duplicate_chunk_ids": {
                    "terms": {
                        "field": "chunk_id",
                        "min_doc_count": 2,
                        "size": size,
                    }
                }
            },
        },
        request_timeout=180,
        )
    except Exception as exc:
        print(f"skip duplicate chunk_id sample for {index_name}: {exc}")
        return None
    return (resp.get("aggregations") or {}).get("duplicate_chunk_ids", {}).get("buckets", [])

def normalize_doc_id_text(value):
    text = str(value or "").strip()
    return text or None

def resolve_doc_id_lookup_batch_size(total_doc_id_count):
    if DOC_ID_LOOKUP_BATCH_SIZE > 0:
        return DOC_ID_LOOKUP_BATCH_SIZE
    if total_doc_id_count <= 10_000:
        return 1000
    if total_doc_id_count <= 100_000:
        return 2000
    return 5000

def resolve_doc_id_delete_batch_size(total_doc_id_count):
    if DOC_ID_DELETE_BATCH_SIZE > 0:
        return DOC_ID_DELETE_BATCH_SIZE
    if total_doc_id_count <= 10_000:
        return 500
    if total_doc_id_count <= 100_000:
        return 1000
    return 2000

def resolve_doc_id_delete_request_timeout(total_doc_id_count):
    if DOC_ID_DELETE_REQUEST_TIMEOUT > 0:
        return max(60, DOC_ID_DELETE_REQUEST_TIMEOUT)
    if total_doc_id_count <= 10_000:
        return 600
    if total_doc_id_count <= 100_000:
        return 1800
    return 3600

def load_doc_id_progress_df():
    files = sorted(
        path for path in list_s3_objects(DOC_ID_PROGRESS_ROOT, recursive=True)
        if str(path).endswith(".jsonl")
    )
    if not files:
        return None
    return (
        spark.read
        .schema(StructType([StructField("doc_id", StringType(), True)]))
        .json([path.replace("s3://", "s3a://") for path in files])
        .where(col("doc_id").isNotNull() & (F.trim(col("doc_id")) != ""))
        .dropDuplicates(["doc_id"])
    )

def write_doc_id_progress(doc_id_df, output_path):
    rows = []
    for row in doc_id_df.select("doc_id").dropDuplicates(["doc_id"]).toLocalIterator():
        doc_id = normalize_doc_id_text(row["doc_id"])
        if doc_id:
            rows.append(json.dumps({"doc_id": doc_id}, ensure_ascii=False))
    if not rows:
        return 0
    payload = "\n".join(rows) + "\n"
    put_s3_object(output_path, payload.encode("utf-8"))
    return len(rows)

def iter_doc_id_batches(doc_id_df, batch_size):
    batch = []
    for row in doc_id_df.select("doc_id").toLocalIterator():
        doc_id = normalize_doc_id_text(row["doc_id"])
        if not doc_id:
            continue
        batch.append(doc_id)
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

def find_existing_doc_ids_batch(client, index_name, doc_ids):
    if not doc_ids:
        return []
    resp = client.search(
        index=index_name,
        body={
            "size": 0,
            "query": {"terms": {"doc_id": doc_ids}},
            "aggs": {
                "existing_doc_ids": {
                    "terms": {
                        "field": "doc_id",
                        "size": max(len(doc_ids), 1),
                    }
                }
            },
        },
        request_timeout=180,
    )
    buckets = ((resp.get("aggregations") or {}).get("existing_doc_ids") or {}).get("buckets", [])
    return [doc_id for doc_id in (normalize_doc_id_text(item.get("key")) for item in buckets) if doc_id]

def materialize_existing_doc_id_df(doc_id_df, batch_size, output_root):
    matched_doc_id_count = 0
    probe_batch_count = 0
    output_file_count = 0
    for probe_batch_count, batch in enumerate(iter_doc_id_batches(doc_id_df, batch_size), start=1):
        existing_doc_ids = find_existing_doc_ids_batch(client, OS_INDEX, batch)
        if existing_doc_ids:
            output_path = f"{output_root.rstrip('/')}/part-{probe_batch_count:08d}.jsonl"
            payload = "\n".join(
                json.dumps({"doc_id": doc_id}, ensure_ascii=False)
                for doc_id in existing_doc_ids
            ) + "\n"
            put_s3_object(output_path, payload.encode("utf-8"))
            matched_doc_id_count += len(existing_doc_ids)
            output_file_count += 1
        if probe_batch_count % 20 == 0:
            print(
                f"doc_id probe progress: batches={probe_batch_count}, "
                f"matched_doc_ids={matched_doc_id_count}, output_files={output_file_count}"
            )
    if output_file_count == 0:
        return None, {
            "probe_batch_count": int(probe_batch_count),
            "existing_doc_id_count": 0,
            "output_file_count": 0,
        }
    existing_doc_id_files = sorted(
        path for path in list_s3_objects(output_root, recursive=True)
        if str(path).endswith(".jsonl")
    )
    existing_doc_id_df = (
        spark.read
        .schema(StructType([StructField("doc_id", StringType(), True)]))
        .json([path.replace("s3://", "s3a://") for path in existing_doc_id_files])
        .dropDuplicates(["doc_id"])
    )
    return existing_doc_id_df, {
        "probe_batch_count": int(probe_batch_count),
        "existing_doc_id_count": int(matched_doc_id_count),
        "output_file_count": int(output_file_count),
    }

def delete_existing_docs_by_doc_id(doc_id_df, batch_size, request_timeout):
    deleted_docs = 0
    delete_batch_count = 0
    requested_doc_id_count = 0
    for delete_batch_count, batch in enumerate(iter_doc_id_batches(doc_id_df, batch_size), start=1):
        resp = client.delete_by_query(
            index=OS_INDEX,
            body={"query": {"terms": {"doc_id": batch}}},
            conflicts="proceed",
            refresh=True,
            wait_for_completion=True,
            request_timeout=request_timeout,
        )
        deleted_docs += int(resp.get("deleted") or 0)
        requested_doc_id_count += len(batch)
        if delete_batch_count % 20 == 0:
            print(
                f"doc_id delete progress: batches={delete_batch_count}, "
                f"requested_doc_ids={requested_doc_id_count}, deleted_docs={deleted_docs}"
            )
    return {
        "delete_batch_count": int(delete_batch_count),
        "requested_doc_id_count": int(requested_doc_id_count),
        "deleted_docs": int(deleted_docs),
    }

def create_index():
    # --- 1. 创建索引（按当前样例字段）---
    validate_os_hosts(OS_HOSTS)
    probe_opensearch(client)
    
    index_exists = client.indices.exists(index=OS_INDEX)
    print(f"index exists before create: {index_exists}")
    if index_exists and RECREATE_INDEX:
        client.indices.delete(index=OS_INDEX, params={"ignore_unavailable": "true"})
        deleted = wait_until_index_absent(client, OS_INDEX, timeout_seconds=INDEX_DELETE_WAIT_SECONDS)
        if not deleted:
            raise RuntimeError(f"删除索引后仍检测到存在: {OS_INDEX}")
        print(f"index deleted: {OS_INDEX}")
    
    BULK_LOAD_SETTINGS = {
        "number_of_shards": NUM_SHARDS,
        "number_of_replicas": 0,
        "refresh_interval": "-1",
        "translog.durability": "async",
        "translog.sync_interval": "120s",
        "translog.flush_threshold_size": "2048mb",
        "index.merge.scheduler.max_thread_count": 4,
        "index.merge.scheduler.max_merge_count": 16,
        "index.codec": "best_compression",
    }
    
    if client.indices.exists(index=OS_INDEX):
        print(f"index already exists, reuse existing index: {OS_INDEX}")
    else:
        client.indices.create(
            index=OS_INDEX,
            body={
                "settings": BULK_LOAD_SETTINGS,
                "mappings": {
                    "dynamic": "strict",
                    "properties": {
                        "record_type": {"type": "keyword"},
                        "chunk_id": {"type": "keyword"},
                        "doc_id": {"type": "keyword"},
                        "title": {
                            "type": "text",
                            "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                        },
                        "content": {"type": "text"},
                        "abstract": {"type": "text"},
                        "url": {
                            "type": "keyword",
                            "doc_values": False,
                            "ignore_above": 4096,
                        },
                        "category": {"type": "keyword"},
                        "source_type": {"type": "keyword"},
                        "lang": {"type": "keyword"},
                        "job_id": {"type": "keyword"},
                        "task_id": {"type": "keyword"},
                        "extra_info": {"type": "object", "dynamic": True},
                        "created_time": {
                            "type": "date",
                            "format": "strict_date_optional_time||epoch_millis",
                        },
                        "updated_time": {
                            "type": "date",
                            "format": "strict_date_optional_time||epoch_millis",
                        },
                        "offset": {"type": "long"},
                        "page_no": {"type": "integer"},
                        "chunk_no": {"type": "integer"},
                        "publication_published_date": {
                            "type": "date",
                            "format": "strict_date_optional_time||yyyy-MM-dd||yyyy-MM||yyyy",
                        },
                        "publication_published_year": {"type": "integer"},
                        "metadata_type": {"type": "keyword"},
                        "publication_venue_type": {"type": "keyword"},
                        "primary_topic": {
                            "type": "text",
                            "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                        },
                        "primary_topic_subfield": {
                            "type": "text",
                            "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                        },
                        "primary_topic_field": {
                            "type": "text",
                            "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                        },
                        "primary_topic_domain": {
                            "type": "text",
                            "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                        },
                        "author": {
                            "type": "text",
                            "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                        },
                        "publication_venue_name_unified": {
                            "type": "text",
                            "fields": {"keyword": {"type": "keyword", "ignore_above": 256}},
                        },
                        "access_is_oa": {"type": "keyword"},
                        "citation_count": {"type": "integer"},
                        "influential_citation_count": {"type": "integer"},
                    },
                },
            },
        )
        print(f"index created: {OS_INDEX}, shards={NUM_SHARDS}")


# --- 3. 写入 ---
def _chunked(iterable, size):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) >= size:
            yield batch
            batch = []
    if batch:
        yield batch

def write_partition(partition):
    log = logging.getLogger("os_writer")
    c = OpenSearch(
        **OS_CONN,
        timeout=180,
        max_retries=2,
        retry_on_timeout=True,
        http_compress=True,
    )

    def _ndjson(docs):
        for d in docs:
            doc_id = str(d.get("chunk_id") or "").strip()
            if not doc_id:
                raise ValueError("chunk_id 不能为空，无法作为 OpenSearch 文档 _id")
            yield json.dumps({BULK_ACTION: {"_index": OS_INDEX, "_id": doc_id}}, ensure_ascii=False)
            yield json.dumps(d, ensure_ascii=False)

    def send(docs, retries=3):
        body = "\n".join(_ndjson(docs)) + "\n"
        resp = None
        for attempt in range(retries + 1):
            try:
                resp = c.bulk(
                    body=body,
                    filter_path=f"errors,items.{BULK_ACTION}.error.type,items.{BULK_ACTION}.error.reason",
                    request_timeout=180,
                )
                if not resp.get("errors"):
                    return len(docs), 0, 0
            except Exception:
                resp = None
            if attempt < retries:
                time.sleep(min(2 ** attempt * 5, 30))
        duplicate_fails = 0
        real_fails = 0
        for it in (resp or {}).get("items", []):
            error = (it.get(BULK_ACTION) or {}).get("error") or {}
            error_type = str(error.get("type") or "")
            error_reason = str(error.get("reason") or "")
            if not error:
                continue
            if error_type == "version_conflict_engine_exception" or "document already exists" in error_reason.lower():
                duplicate_fails += 1
            else:
                real_fails += 1
        written = len(docs) - duplicate_fails - real_fails
        return written, duplicate_fails, real_fails

    rows = (
        {k: v for k, v in row.asDict(recursive=True).items() if v is not None}
        for row in partition
    )

    written = duplicate = real_failed = 0
    t0 = time.time()
    for batch in _chunked(rows, BULK_SIZE):
        ok, dup, fail = send(batch)
        written += ok
        duplicate += dup
        real_failed += fail

    elapsed = max(time.time() - t0, 1)
    log.warning(
        "done: %d written, %d duplicate, %d real_failed, %.0f docs/s, %.1f s",
        written,
        duplicate,
        real_failed,
        written / elapsed,
        elapsed,
    )
    return {"written": int(written), "duplicate": int(duplicate), "real_failed": int(real_failed)}




In [ ]:
# 这里直接使用当前输入的目标字段名。
# raw_schema = """
# record_type STRING,
# chunk_id STRING,
# doc_id STRING,
# title STRING,
# content STRING,
# abstract STRING,
# url STRING,
# category STRING,
# source_type STRING,
# lang STRING,
# job_id STRING,
# task_id STRING,
# extra_info MAP<STRING,STRING>,
# created_time STRING,
# updated_time STRING,
# offset BIGINT,
# page_no BIGINT,
# chunk_no BIGINT,
# publication_published_date STRING,
# publication_published_year BIGINT,
# metadata_type STRING,
# publication_venue_type STRING,
# primary_topic STRING,
# primary_topic_subfield STRING,
# primary_topic_field STRING,
# primary_topic_domain STRING,
# author ARRAY<STRING>,
# publication_venue_name_unified STRING,
# access_is_oa STRING,
# citation_count BIGINT,
# influential_citation_count BIGINT
# """

raw_schema = StructType([
    StructField("record_type", StringType(), nullable=True),
    StructField("chunk_id", StringType(), nullable=True),
    StructField("doc_id", StringType(), nullable=True),
    StructField("title", StringType(), nullable=True),
    StructField("content", StringType(), nullable=True),
    StructField("abstract", StringType(), nullable=True),
    StructField("url", StringType(), nullable=True),
    StructField("category", StringType(), nullable=True),
    StructField("source_type", StringType(), nullable=True),
    StructField("lang", StringType(), nullable=True),
    StructField("job_id", StringType(), nullable=True),
    StructField("task_id", StringType(), nullable=True),
    StructField("extra_info", MapType(StringType(), StringType(), True), True),
    StructField("created_time", StringType(), nullable=True),
    StructField("updated_time", StringType(), nullable=True),
    StructField("offset", LongType(), nullable=False),
    StructField("page_no", LongType(), nullable=False),
    StructField("chunk_no", LongType(), nullable=False),
    StructField("publication_published_date", StringType(), nullable=True),
    StructField("publication_published_year", LongType(), nullable=False),
    StructField("metadata_type", StringType(), nullable=True),
    StructField("publication_venue_type", StringType(), nullable=True),
    StructField("primary_topic", StringType(), nullable=True),
    StructField("primary_topic_subfield", StringType(), nullable=True),
    StructField("primary_topic_field", StringType(), nullable=True),
    StructField("primary_topic_domain", StringType(), nullable=True),
    StructField("author", ArrayType(StringType(), True), True),
    StructField("publication_venue_name_unified", StringType(), nullable=True),
    StructField("access_is_oa", StringType(), nullable=True),
    StructField("citation_count", LongType(), nullable=False),
    StructField("influential_citation_count", LongType(), nullable=False)
])



In [ ]:
# raw_df = read_any_path(spark, input_path, config)
# # raw_df.printSchema()
# raw_df.select("value").show(3, truncate=False)


# read

In [ ]:
# --- 1. 创建索引（按当前样例字段）---
create_index()


In [ ]:
input_path

In [ ]:
# --- 2. 准备数据 ---
# 指定schema读取json
progress_payload = load_progress_payload(INPUT_PROGRESS_PATH)
progress_prefix = f"{INPUT_PROGRESS_ROOT.rstrip('/')}/"
all_input_files = sorted(
    path for path in list_s3_objects(input_path, recursive=True)
    if str(path).endswith('.json') and not str(path).startswith(progress_prefix)
)
print(f"all_input_files={len(all_input_files)}")
if not all_input_files:
    raise RuntimeError("input_path 下没有可写入 OpenSearch 的 .json 输入文件")

parsed_df = spark.read.option("recursiveFileLookup", "false") \
    .option("pathGlobFilter", "*.json") \
    .schema(raw_schema) \
    .json([path.replace("s3://", "s3a://") for path in all_input_files]) \
    .repartition(READ_PARTITIONS)
print(f"input files ready: {len(all_input_files)}")
# # parsed_df = raw_df.select(from_json(col("value"), raw_schema).alias("d")).select("d.*")


In [ ]:
source_df = parsed_df.limit(TEST_SAMPLE_ROWS) if TEST_SAMPLE_ROWS else parsed_df

base_df = (
    source_df.select(
        col("record_type"),
        F.trim(col("chunk_id")).alias("chunk_id"),
        F.trim(col("doc_id")).alias("doc_id"),
        lower(col("title")).alias("title"),
        col("content"),
        col("abstract"),
        col("url"),
        col("category"),
        col("source_type"),
        col("lang"),
        col("job_id"),
        col("task_id"),
        parse_extra_info_udf(col("extra_info")).alias("extra_info"),
        col("created_time"),
        col("updated_time"),
        col("offset"),
        col("page_no"),
        col("chunk_no"),
        col("publication_published_date"),
        col("publication_published_year"),
        col("metadata_type"),
        lower(col("publication_venue_type")).alias("publication_venue_type"),
        lower(col("primary_topic")).alias("primary_topic"),
        lower(col("primary_topic_subfield")).alias("primary_topic_subfield"),
        lower(col("primary_topic_field")).alias("primary_topic_field"),
        lower(col("primary_topic_domain")).alias("primary_topic_domain"),
        transform(col("author"), lambda x: lower(x)).alias("author"),
        lower(col("publication_venue_name_unified")).alias("publication_venue_name_unified"),
        col("access_is_oa"),
        col("citation_count"),
        col("influential_citation_count"),
    )
    .cache()
)

missing_chunk_id_rows = base_df.where(col("chunk_id").isNull() | (F.trim(col("chunk_id")) == "")).count()
if missing_chunk_id_rows:
    raise RuntimeError(
        f"检测到 {missing_chunk_id_rows} 条 chunk_id 为空的数据，无法把 chunk_id 作为 OpenSearch 文档 _id"
    )
missing_doc_id_rows = base_df.where(col("doc_id").isNull() | (F.trim(col("doc_id")) == "")).count()
if missing_doc_id_rows:
    raise RuntimeError(
        f"检测到 {missing_doc_id_rows} 条 doc_id 为空的数据，无法按 doc_id 判断跳过或重建"
    )
input_doc_id_df = (
    base_df
    .select("doc_id")
    .where(col("doc_id").isNotNull() & (F.trim(col("doc_id")) != ""))
    .dropDuplicates(["doc_id"])
    .cache()
)
input_unique_doc_id_count = input_doc_id_df.count()
resolved_doc_id_lookup_batch_size = resolve_doc_id_lookup_batch_size(input_unique_doc_id_count)
resolved_doc_id_delete_batch_size = resolve_doc_id_delete_batch_size(input_unique_doc_id_count)
resolved_doc_id_delete_request_timeout = resolve_doc_id_delete_request_timeout(input_unique_doc_id_count)
completed_doc_id_df = load_doc_id_progress_df()
completed_doc_id_count = 0
candidate_df = base_df
candidate_doc_id_df = input_doc_id_df
if completed_doc_id_df is not None:
    completed_doc_id_df = completed_doc_id_df.cache()
    completed_doc_id_count = completed_doc_id_df.count()
    candidate_df = base_df.join(completed_doc_id_df, "doc_id", "left_anti")
    candidate_doc_id_df = candidate_df.select("doc_id").dropDuplicates(["doc_id"]).cache()
else:
    candidate_doc_id_df = input_doc_id_df
existing_doc_id_count = 0
doc_id_probe_batch_count = 0
doc_id_probe_output_file_count = 0
deleted_doc_count = 0
doc_id_delete_batch_count = 0
doc_id_progress_written_count = 0
write_source_df = candidate_df

if SKIP_EXISTING_DOC_IDS:
    existing_doc_id_df, doc_id_probe_stats = materialize_existing_doc_id_df(
        candidate_doc_id_df,
        resolved_doc_id_lookup_batch_size,
        CURRENT_DOC_ID_PROBE_ROOT,
    )
    doc_id_probe_batch_count = int(doc_id_probe_stats.get("probe_batch_count") or 0)
    doc_id_probe_output_file_count = int(doc_id_probe_stats.get("output_file_count") or 0)
    if existing_doc_id_df is not None:
        existing_doc_id_df = existing_doc_id_df.cache()
        existing_doc_id_count = existing_doc_id_df.count()
        write_source_df = candidate_df.join(existing_doc_id_df, "doc_id", "left_anti")
    else:
        existing_doc_id_count = 0
else:
    if RECREATE_INDEX:
        print("RECREATE_INDEX=True：索引已重建，跳过按 doc_id 删除历史数据")
    else:
        delete_stats = delete_existing_docs_by_doc_id(
            candidate_doc_id_df,
            resolved_doc_id_delete_batch_size,
            resolved_doc_id_delete_request_timeout,
        )
        deleted_doc_count = int(delete_stats.get("deleted_docs") or 0)
        doc_id_delete_batch_count = int(delete_stats.get("delete_batch_count") or 0)

df = write_source_df.repartition(WRITE_PARTITIONS).cache()
prepared_rows = df.count()
if prepared_rows == 0:
    raise RuntimeError("按 doc_id 过滤后没有待写入 OpenSearch 的数据")

print(f"rows prepared for write: {prepared_rows}")
print(f"unique input doc_id count: {input_unique_doc_id_count}")
print(f"completed doc_id count from local progress: {completed_doc_id_count}")
print(f"skip_existing_doc_ids: {SKIP_EXISTING_DOC_IDS}")
print(f"resolved doc_id lookup batch size: {resolved_doc_id_lookup_batch_size}")
print(f"resolved doc_id delete batch size: {resolved_doc_id_delete_batch_size}")
print(f"resolved doc_id delete timeout seconds: {resolved_doc_id_delete_request_timeout}")
print(f"existing doc_id count in OpenSearch: {existing_doc_id_count}")
print(f"docs deleted by doc_id before reimport: {deleted_doc_count}")
print("写入完成后，如果 RECREATE_INDEX=True，则重复数据数量 = prepared_rows - indexed_docs_after_refresh")
if TEST_SAMPLE_ROWS:
    preview_docs = [row.asDict(recursive=True) for row in df.limit(TEST_SAMPLE_ROWS).collect()]
    print(json.dumps(preview_docs, ensure_ascii=False, indent=2))


# write & check

In [ ]:
# --- 3. 写入 ---
stats_df = df.rdd.mapPartitions(lambda partition: [Row(**write_partition(partition))]).toDF()
stats_row = stats_df.select(
    F.sum(F.col("written")).alias("written"),
    F.sum(F.col("duplicate")).alias("duplicate"),
    F.sum(F.col("real_failed")).alias("real_failed"),
).collect()[0]
written_rows = int(stats_row["written"] or 0)
duplicate_rows = int(stats_row["duplicate"] or 0)
real_failed_rows = int(stats_row["real_failed"] or 0)
written_doc_id_df = df.select("doc_id").dropDuplicates(["doc_id"])
doc_id_progress_written_count = write_doc_id_progress(written_doc_id_df, CURRENT_DOC_ID_PROGRESS_PATH)
stats_df.unpersist() if hasattr(stats_df, 'unpersist') else None
df.unpersist()
base_df.unpersist()
input_doc_id_df.unpersist()
candidate_doc_id_df.unpersist() if candidate_doc_id_df is not input_doc_id_df else None
completed_doc_id_df.unpersist() if completed_doc_id_df is not None else None
print(f"bulk import finished, rows attempted: {prepared_rows}, written={written_rows}, duplicate={duplicate_rows}, real_failed={real_failed_rows}")
if real_failed_rows > 0:
    raise RuntimeError(f"OpenSearch bulk 写入存在真实失败记录，written={written_rows}, duplicate={duplicate_rows}, real_failed={real_failed_rows}，不推进 progress")
save_progress_payload(
    INPUT_PROGRESS_PATH,
    {
        'input_root': input_path,
        'input_file_count': len(all_input_files),
        'skip_existing_doc_ids': SKIP_EXISTING_DOC_IDS,
        'prepared_rows': prepared_rows,
        'input_unique_doc_id_count': input_unique_doc_id_count,
        'completed_doc_id_count': completed_doc_id_count,
        'existing_doc_id_count': existing_doc_id_count,
        'resolved_doc_id_lookup_batch_size': resolved_doc_id_lookup_batch_size,
        'doc_id_probe_batch_count': doc_id_probe_batch_count,
        'doc_id_probe_output_file_count': doc_id_probe_output_file_count,
        'deleted_doc_count': deleted_doc_count,
        'resolved_doc_id_delete_batch_size': resolved_doc_id_delete_batch_size,
        'resolved_doc_id_delete_request_timeout': resolved_doc_id_delete_request_timeout,
        'doc_id_delete_batch_count': doc_id_delete_batch_count,
        'doc_id_progress_path': CURRENT_DOC_ID_PROGRESS_PATH,
        'doc_id_progress_written_count': doc_id_progress_written_count,
        'written_rows': written_rows,
        'duplicate_rows': duplicate_rows,
        'real_failed_rows': real_failed_rows,
        'run_id': RUN_ID,
        'updated_at': datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z'),
    },
)

# --- 4. 回查验证 ---
client.indices.refresh(index=OS_INDEX)
count_resp = client.count(index=OS_INDEX)
indexed_docs_after_refresh = int(count_resp["count"])
print(f"indexed docs after refresh: {indexed_docs_after_refresh}")

if RECREATE_INDEX:
    duplicate_doc_rows = prepared_rows - indexed_docs_after_refresh
    if duplicate_doc_rows < 0:
        raise RuntimeError(
            f"indexed_docs_after_refresh({indexed_docs_after_refresh}) > prepared_rows({prepared_rows})，请检查是否存在并发写入或历史数据残留"
        )
    print(f"source duplicate doc rows by chunk_id: {duplicate_doc_rows}")
    print(f"source unique chunk_id rows: {indexed_docs_after_refresh}")
else:
    print("RECREATE_INDEX=False：当前索引可能含历史数据，无法仅凭最终 count 计算本次重复量")

duplicate_buckets = sample_duplicate_chunk_ids(client, OS_INDEX, size=10)
if duplicate_buckets is not None:
    print(f"duplicate chunk_id buckets in OpenSearch after write: {len(duplicate_buckets)}")
    print(json.dumps(duplicate_buckets, ensure_ascii=False, indent=2))

verify_resp = client.search(
    index=OS_INDEX,
    body={
        "size": 10,
        "sort": ["_doc"],
        "query": {"match_all": {}},
    },
)
verify_docs = [hit.get("_source", {}) for hit in verify_resp.get("hits", {}).get("hits", [])]
print(json.dumps(verify_docs, ensure_ascii=False, indent=2))


In [ ]:
# --- 5. 切换为只读查询优化设置 ---
# if TEST_SAMPLE_ROWS:
#     print("测试模式：已跳过 force merge 和只读优化设置。确认样本无误后，将 TEST_SAMPLE_ROWS 改成 None 或 0 再全量导入。")
# else:
#     client.indices.refresh(index=OS_INDEX, params={"wait_for_completion": "false"})
#     print("refresh 已在后台启动，通过 GET /_tasks?actions=*refresh&detailed 观察进度")

#     client.indices.forcemerge(
#         index=OS_INDEX,
#         max_num_segments=1,
#         flush=True,
#         params={"wait_for_completion": "false"},
#     )
#     print("force merge 已在后台启动，通过 GET /_cat/segments/<index>?v 观察 segment 数量收敛")

#     READONLY_SETTINGS = {
#         "index": {
#             "number_of_replicas": 1,
#             "refresh_interval": "30s",
#             "translog.durability": "request",
#             "merge.scheduler.max_thread_count": 1,
#             "merge.scheduler.max_merge_count": 1,
#         }
#     }

#     client.indices.put_settings(index=OS_INDEX, body=READONLY_SETTINGS)
#     print("readonly settings applied")


# force merge

In [ ]:
# 大索引 force merge 可能超过网关超时，默认改为异步启动。
FORCE_MERGE_ASYNC = True
FORCE_MERGE_TASK_ID = None
# --- 5A. 启动 force merge ---
if TEST_SAMPLE_ROWS:
    print("测试模式：已跳过 force merge 和只读优化设置。确认样本无误后，将 TEST_SAMPLE_ROWS 改成 None 或 0 再全量导入。")
else:
    client.indices.refresh(index=OS_INDEX)
    print("refresh 已完成")

    if FORCE_MERGE_ASYNC:
        force_merge_resp = client.indices.forcemerge(
            index=OS_INDEX,
            max_num_segments=1,
            flush=True,
            params={"wait_for_completion": "false"},
        )
        FORCE_MERGE_TASK_ID = force_merge_resp.get("task")
        print(f"force merge 已异步启动，task_id={FORCE_MERGE_TASK_ID}")
        print("merge 完成后再执行下一个单元应用只读优化设置")
    else:
        client.indices.forcemerge(
            index=OS_INDEX,
            max_num_segments=1,
            flush=True,
        )
        FORCE_MERGE_TASK_ID = None
        print("force merge 已同步完成，可直接执行下一个单元")

# check progress

In [ ]:
# --- 5B. merge 完成后执行只读优化设置 ---
READONLY_SETTINGS = {
    "index": {
        "number_of_replicas": 1,
        "refresh_interval": "30s",
        "translog.durability": "request",
        "merge.scheduler.max_thread_count": 1,
        "merge.scheduler.max_merge_count": 1,
    }
}

if TEST_SAMPLE_ROWS:
    print("测试模式：跳过只读优化设置")
elif FORCE_MERGE_ASYNC and FORCE_MERGE_TASK_ID:
    try:
        task_resp = client.tasks.get(task_id=FORCE_MERGE_TASK_ID)
        # 校验opensearch设置只读模式是否完成，completed: True 表示完成，否则未完成
        completed = bool(task_resp.get("completed"))
        print(json.dumps(task_resp, ensure_ascii=False, indent=2))
        if not completed:
            print(f"force merge 尚未完成，请稍后重试本单元。task_id={FORCE_MERGE_TASK_ID}")
    except Exception as exc:
        msg = str(exc)
        if "404" in msg or "resource_not_found_exception" in msg:
            print(f"未在 Tasks API 中找到 task_id={FORCE_MERGE_TASK_ID}，按已完成处理")
        else:
            raise
    client.indices.put_settings(index=OS_INDEX, body=READONLY_SETTINGS)
    print("readonly settings applied")
else:
    client.indices.put_settings(index=OS_INDEX, body=READONLY_SETTINGS)
    print("readonly settings applied")
